# Clickbait Detection — IndoBERT Fixed Notebook (v2 — All Bugs Patched)

**Paper:** *Clickbait Detection in Indonesia Headline News Using IndoBERT and RoBERTa*  
**Journal:** Jurnal Riset Informatika, Vol. 5 No. 3, June 2023

**Metodologi:**
1. Load dataset `all_agree.csv`.
2. Membuat empat setting eksperimen: Imbalance, Balance, Imbalance + Augment, Balance + Augment.
3. Fine-tune IndoBERT (`indobenchmark/indobert-base-p1`).
4. Evaluasi menggunakan Precision, Recall, F1-score (weighted + macro), dan Accuracy.

**Struktur folder utama di Google Drive:**
```text
MyDrive/NLPAOL_V3/
  data_1/annotated/combined/csv/all_agree.csv
  outputs/
    indobert_model/
    final_results/
```

**Perbaikan dalam versi ini (v2):**

| # | Lokasi | Masalah | Perbaikan |
|---|--------|---------|-----------|
| 1 | Cell 7c `eda_augment` | `SynonymAug(aug_src="wordnet")` menggunakan **English WordNet** → menyuntikkan kata bahasa Inggris ke teks Indonesia, merusak semantik secara diam-diam | Dihapus sepenuhnya. Hanya `RandomWordAug(swap)` dan `RandomWordAug(delete)` yang dipertahankan — keduanya tidak bergantung pada bahasa |
| 2 | Cell 6 setelah `make_splits` | Tidak ada pengecekan apakah train/val/test benar-benar disjoint | Ditambahkan 3 `assert` untuk mendeteksi kebocoran data akibat refactor di masa depan |
| 3 | Cell 9 `compute_metrics` | Hanya weighted F1 — menyembunyikan performa buruk pada kelas minoritas | Ditambahkan `f1_macro` ke dict return |
| 4 | Cell 10 `train_and_evaluate` | `result` dict tidak menyertakan macro F1 | Ditambahkan `test_f1_macro` ke result |
| 5 | Setelah pre-processing | Tidak ada analisis distribusi panjang token | Ditambahkan Cell 5b untuk cek apakah `MAX_LENGTH=96` cukup |
| 6 | Cell 4 imports | `nltk.download("wordnet")` tidak diperlukan setelah SynonymAug dihapus | Dihapus |


## 1. Install Dependencies

In [ ]:
# Run this cell first.
# After it finishes, restart runtime: Runtime > Restart runtime

!pip uninstall -y peft transformers accelerate tokenizers
!pip install -q transformers==4.48.3 accelerate==1.3.0 tokenizers==0.21.0 datasets evaluate nlpaug scikit-learn

print("Package install selesai. Sekarang lakukan Runtime > Restart runtime, lalu lanjut dari Cell 4.")


Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.6/336.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/NLPAOL_V3")

# Expected location
DATA_DIR = PROJECT_DIR / "data_1" / "annotated" / "combined" / "csv"
csv_path = DATA_DIR / "all_agree.csv"

# Auto-search if file is not found in expected location
if not csv_path.exists():
    print("CSV tidak ditemukan di expected path. Searching inside PROJECT_DIR...")
    matches = list(PROJECT_DIR.rglob("all_agree.csv"))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"all_agree.csv tidak ditemukan di dalam folder: {PROJECT_DIR}"
        )

    csv_path = matches[0]
    DATA_DIR = csv_path.parent

OUTPUT_DIR  = PROJECT_DIR / "outputs_indobert" / "indobert_model"
RESULTS_DIR = PROJECT_DIR / "outputs_indobert" / "final_results"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR :", PROJECT_DIR)
print("DATA_DIR    :", DATA_DIR)
print("CSV PATH    :", csv_path)
print("CSV EXISTS  :", csv_path.exists())


Mounted at /content/drive
PROJECT_DIR : /content/drive/MyDrive/NLPAOL_V3
DATA_DIR    : /content/drive/MyDrive/NLPAOL_V3/data_1/annotated/combined/csv
CSV PATH    : /content/drive/MyDrive/NLPAOL_V3/data_1/annotated/combined/csv/all_agree.csv
CSV EXISTS  : True


In [ ]:
import os, re, random, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import nltk


from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.utils import resample

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed,
)

from datasets import Dataset
import nlpaug.augmenter.word as naw

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

print("Transformers version:", transformers.__version__)
print("Torch version       :", torch.__version__)
print("CUDA available      :", torch.cuda.is_available())


Transformers version: 4.48.3
Torch version       : 2.11.0+cu128
CUDA available      : True


In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

MAX_LENGTH   = 96

# Safer for Colab T4/G4. Effective batch size = BATCH_SIZE * gradient_accumulation_steps.
BATCH_SIZE   = 8
EPOCHS       = 10
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
PATIENCE     = 3
SEED         = 42

LABEL2ID = {
    "non-clickbait": 0,
    "clickbait": 1,
}

ID2LABEL = {
    0: "non-clickbait",
    1: "clickbait",
}

TEXT_COL  = "title"
LABEL_COL = "label"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(SEED)

print(f"Model      : {MODEL_NAME}")
print(f"Device     : {DEVICE}")
print(f"Data dir   : {DATA_DIR}")


Model      : indobenchmark/indobert-base-p1
Device     : cuda
Data dir   : /content/drive/MyDrive/NLPAOL_V3/data_1/annotated/combined/csv


In [ ]:
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

display(df.head())

if TEXT_COL not in df.columns:
    raise ValueError(f"Kolom text '{TEXT_COL}' tidak ditemukan. Kolom tersedia: {df.columns.tolist()}")

if LABEL_COL not in df.columns:
    raise ValueError(f"Kolom label '{LABEL_COL}' tidak ditemukan. Kolom tersedia: {df.columns.tolist()}")

print("\nMissing values:")
print(df[[TEXT_COL, LABEL_COL]].isnull().sum())

print("\nLabel distribution:")
print(df[LABEL_COL].value_counts(dropna=False))


Dataset shape: (8613, 3)
Columns: ['title', 'label', 'label_score']


,title,label,label_score
0,"Masuk Radar Pilwalkot Medan, Menantu Jokowi Be...",non-clickbait,0
1,Malaysia Sudutkan RI: Isu Kabut Asap hingga In...,non-clickbait,0
2,Viral! Driver Ojol di Bekasi Antar Pesanan Mak...,clickbait,1
3,"Kemensos Salurkan Rp 7,3 M bagi Korban Kerusuh...",non-clickbait,0
4,MPR: Amandemen UUD 1945 Tak Akan Melebar ke Ma...,non-clickbait,0



Missing values:
title    0
label    0
dtype: int64

Label distribution:
label
non-clickbait    5297
clickbait        3316
Name: count, dtype: int64


## 5. Text Pre-processing

In [ ]:
def clean_text(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^\w\s.,!?-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_label(label):
    label = str(label).strip().lower()
    label = label.replace("_", "-")
    label = re.sub(r"\s+", "-", label)

    if label in ["clickbait", "cb", "1"]:
        return "clickbait"
    if label in ["non-clickbait", "nonclickbait", "non-click-bait", "noncb", "not-clickbait", "0"]:
        return "non-clickbait"
    return label


df_raw = df.copy()
df_raw = df_raw[[TEXT_COL, LABEL_COL]].dropna().copy()

df_raw[TEXT_COL] = df_raw[TEXT_COL].astype(str)
df_raw[LABEL_COL] = df_raw[LABEL_COL].apply(normalize_label)

df_raw["title_clean"] = df_raw[TEXT_COL].apply(clean_text)
df_raw["label_id"] = df_raw[LABEL_COL].map(LABEL2ID)

invalid_labels = df_raw[df_raw["label_id"].isna()][LABEL_COL].unique()

if len(invalid_labels) > 0:
    raise ValueError(f"Ada label yang tidak sesuai LABEL2ID: {invalid_labels}")

df_raw["label_id"] = df_raw["label_id"].astype(int)

# Remove empty titles after cleaning
df_raw = df_raw[df_raw["title_clean"].str.len() > 0].reset_index(drop=True)

print("Final dataset shape:", df_raw.shape)
display(df_raw[["title_clean", LABEL_COL, "label_id"]].head())

print("\nFinal label distribution:")
print(df_raw[LABEL_COL].value_counts())


Final dataset shape: (8613, 4)


,title_clean,label,label_id
0,"Masuk Radar Pilwalkot Medan, Menantu Jokowi Be...",non-clickbait,0
1,Malaysia Sudutkan RI Isu Kabut Asap hingga Inv...,non-clickbait,0
2,Viral! Driver Ojol di Bekasi Antar Pesanan Mak...,clickbait,1
3,"Kemensos Salurkan Rp 7,3 M bagi Korban Kerusuh...",non-clickbait,0
4,MPR Amandemen UUD 1945 Tak Akan Melebar ke Man...,non-clickbait,0



Final label distribution:
label
non-clickbait    5297
clickbait        3316
Name: count, dtype: int64


## 5b. Token Length Check (Confirm MAX_LENGTH Sufficiency)


In [ ]:
# ── FIX: Verify MAX_LENGTH=96 covers the distribution without silent truncation ──
token_lengths = df_raw["title_clean"].apply(lambda t: len(t.split()))

print("Word-count distribution of title_clean:")
print(token_lengths.describe().round(1))
print(f"\nMax words in dataset : {token_lengths.max()}")
print(f"99th percentile      : {token_lengths.quantile(0.99):.0f} words")
print(f"MAX_LENGTH set to    : {MAX_LENGTH} tokens")

pct_truncated = (token_lengths > MAX_LENGTH).mean() * 100
if pct_truncated > 1.0:
    print(f"\n⚠️  WARNING: ~{pct_truncated:.1f}% of rows may be truncated. Consider raising MAX_LENGTH to 128.")
else:
    print(f"\n✓ Only {pct_truncated:.1f}% of rows may be truncated — MAX_LENGTH={MAX_LENGTH} is sufficient.")


Word-count distribution of title_clean:
count    8613.0
mean        9.7
std         2.3
min         2.0
25%         8.0
50%         9.0
75%        11.0
max        19.0
Name: title_clean, dtype: float64

Max words in dataset : 19
99th percentile      : 15 words
MAX_LENGTH set to    : 96 tokens

✓ Only 0.0% of rows may be truncated — MAX_LENGTH=96 is sufficient.


## 6. Train / Val / Test Split (80:10:10)

In [ ]:
def make_splits(df_input, seed=SEED):
    # 10% test
    train_val, test = train_test_split(
        df_input,
        test_size=0.10,
        stratify=df_input["label_id"],
        random_state=seed,
    )

    # 10% validation from total data = 0.1111 of remaining 90%
    train, val = train_test_split(
        train_val,
        test_size=0.1111,
        stratify=train_val["label_id"],
        random_state=seed,
    )

    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )

# ── Stamp a stable row identity BEFORE splitting ──
df_raw["_row_id"] = range(len(df_raw))

train_raw, val_raw, test_raw = make_splits(df_raw)

print(f"Train: {len(train_raw):,}")
print(f"Val  : {len(val_raw):,}")
print(f"Test : {len(test_raw):,}")

print("\nTrain distribution:")
print(train_raw[LABEL_COL].value_counts())
print("\nVal distribution:")
print(val_raw[LABEL_COL].value_counts())
print("\nTest distribution:")
print(test_raw[LABEL_COL].value_counts())

# ── FIX: Check original row identity, not reset integer index ──
assert len(set(train_raw["_row_id"]) & set(val_raw["_row_id"]))  == 0, "Train/Val overlap!"
assert len(set(train_raw["_row_id"]) & set(test_raw["_row_id"])) == 0, "Train/Test overlap!"
assert len(set(val_raw["_row_id"])   & set(test_raw["_row_id"])) == 0, "Val/Test overlap!"

print("✓ No overlap between splits")

Train: 6,889
Val  : 862
Test : 862

Train distribution:
label
non-clickbait    4237
clickbait        2652
Name: count, dtype: int64

Val distribution:
label
non-clickbait    530
clickbait        332
Name: count, dtype: int64

Test distribution:
label
non-clickbait    530
clickbait        332
Name: count, dtype: int64
✓ No overlap between splits


## 7a. Dataset Setting: Imbalance

In [ ]:
train_imbalance = train_raw.copy()

print("[Imbalance] Distribusi train:")
print(train_imbalance[LABEL_COL].value_counts())


[Imbalance] Distribusi train:
label
non-clickbait    4237
clickbait        2652
Name: count, dtype: int64


## 7b. Dataset Setting: Balance (Bootstrapping)

In [ ]:
def balance_by_bootstrap(df_input, seed=SEED):
    majority = df_input[LABEL_COL].value_counts().idxmax()
    minority = df_input[LABEL_COL].value_counts().idxmin()

    n_maj = (df_input[LABEL_COL] == majority).sum()

    df_maj = df_input[df_input[LABEL_COL] == majority]
    df_min = df_input[df_input[LABEL_COL] == minority]

    df_min_up = resample(
        df_min,
        replace=True,
        n_samples=n_maj,
        random_state=seed,
    )

    balanced_df = (
        pd.concat([df_maj, df_min_up])
        .sample(frac=1, random_state=seed)
        .reset_index(drop=True)
    )

    return balanced_df

train_balance = balance_by_bootstrap(train_raw)

print("[Balance] Distribusi setelah bootstrapping:")
print(train_balance[LABEL_COL].value_counts())


[Balance] Distribusi setelah bootstrapping:
label
clickbait        4237
non-clickbait    4237
Name: count, dtype: int64


## 7c. Dataset Setting: Augmented (EDA)

In [ ]:
# ── FIX: Removed English WordNet SynonymAug — it injects English words into
#         Indonesian text, causing silent semantic corruption.
#         Only language-agnostic operations (swap, delete) are used.
def eda_augment(texts, seed=SEED):
    """
    EDA using only language-agnostic operations:
    - Random word swap
    - Random word deletion
    (SynonymAug removed — WordNet is English-only and corrupts Indonesian text)
    """
    random.seed(seed)
    np.random.seed(seed)

    aug_swap   = naw.RandomWordAug(action="swap",   aug_p=0.1)
    aug_delete = naw.RandomWordAug(action="delete", aug_p=0.1)

    ops = [aug_swap, aug_delete]
    out = []

    for text in texts:
        op = random.choice(ops)
        try:
            res = op.augment(str(text), n=1)
            out.append(res[0] if isinstance(res, list) and len(res) > 0 else text)
        except Exception:
            out.append(text)

    return out

print("Menjalankan EDA pada balanced training set...")

aug_texts = eda_augment(train_balance["title_clean"].tolist())

df_aug = train_balance.copy()
df_aug["title_clean"] = aug_texts

train_augmented = (
    pd.concat([train_balance, df_aug])
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

print("[Augmented] Distribusi setelah EDA:")
print(train_augmented[LABEL_COL].value_counts())
print(f"Total: {len(train_augmented):,}")


Menjalankan EDA pada balanced training set...
[Augmented] Distribusi setelah EDA:
label
clickbait        8474
non-clickbait    8474
Name: count, dtype: int64
Total: 16,948


## 8. Tokenizer & HuggingFace Dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def df_to_hf_dataset(df_input):
    temp_df = df_input[["title_clean", "label_id"]].copy()
    temp_df = temp_df.rename(columns={"label_id": "labels"})

    ds = Dataset.from_pandas(temp_df, preserve_index=False)

    ds = ds.map(
        lambda batch: tokenizer(
            batch["title_clean"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        ),
        batched=True,
        batch_size=512,
    )

    ds = ds.remove_columns(["title_clean"])
    ds.set_format("torch")

    return ds

val_ds = df_to_hf_dataset(val_raw)
test_ds = df_to_hf_dataset(test_raw)

print("Val dan Test dataset tokenized OK")
print(val_ds)
print(test_ds)


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/862 [00:00<?, ? examples/s]

Map:   0%|          | 0/862 [00:00<?, ? examples/s]

Val dan Test dataset tokenized OK
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 862
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 862
})


## 9. Metrics

In [ ]:
# ── FIX: Added macro F1 — weighted F1 alone can hide poor minority-class performance ──
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    preds = np.argmax(logits, axis=-1)

    return {
        "f1":        f1_score(labels, preds, average="weighted", zero_division=0),
        "f1_macro":  f1_score(labels, preds, average="macro",    zero_division=0),  # NEW
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall":    recall_score(labels, preds, average="weighted", zero_division=0),
        "accuracy":  float((preds == labels).mean()),
    }


## 10. Training Function

In [ ]:
def train_and_evaluate(train_df, val_ds, test_ds, experiment_name, model_name=MODEL_NAME):
    print(f"\n{'='*60}")
    print(f"Eksperimen : {experiment_name}")
    print(f"Train rows : {len(train_df):,}")
    print(train_df[LABEL_COL].value_counts().to_string())
    print("="*60)

    set_seed(SEED)

    train_ds = df_to_hf_dataset(train_df)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    run_dir = OUTPUT_DIR / experiment_name
    run_dir.mkdir(parents=True, exist_ok=True)

    args = TrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        gradient_accumulation_steps=2,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=50,
        seed=SEED,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
    )

    trainer.train()

    val_metrics = trainer.evaluate(val_ds)
    test_out = trainer.predict(test_ds)

    test_labels = test_out.label_ids
    test_logits = test_out.predictions

    if isinstance(test_logits, tuple):
        test_logits = test_logits[0]

    test_preds = np.argmax(test_logits, axis=-1)

    report = classification_report(
        test_labels,
        test_preds,
        target_names=["non-clickbait", "clickbait"],
        digits=4,
        zero_division=0,
    )

    print("\n── TEST CLASSIFICATION REPORT ──")
    print(report)

    report_path = RESULTS_DIR / f"{experiment_name}_report.txt"
    report_path.write_text(
        f"""Experiment: {experiment_name}
Model: {model_name}

{report}
""",
        encoding="utf-8",
    )

    # ── FIX: Added test_f1_macro to reveal per-class performance gaps ──
    result = {
        "experiment":    experiment_name,
        "model":         model_name,
        "train_rows":    len(train_df),
        "val_f1":        round(val_metrics.get("eval_f1", 0), 4),
        "test_precision": round(
            precision_score(test_labels, test_preds, average="weighted", zero_division=0), 4,
        ),
        "test_recall":   round(
            recall_score(test_labels, test_preds, average="weighted", zero_division=0), 4,
        ),
        "test_f1":       round(
            f1_score(test_labels, test_preds, average="weighted", zero_division=0), 4,
        ),
        "test_f1_macro": round(
            f1_score(test_labels, test_preds, average="macro", zero_division=0), 4,  # NEW
        ),
    }

    del model
    del trainer
    del train_ds

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


In [ ]:
import datasets.config as datasets_config

datasets_config.TORCHVISION_AVAILABLE = False

## 11. Jalankan Eksperimen 1 — Imbalance

In [ ]:
all_results = []

res = train_and_evaluate(
    train_imbalance,
    val_ds,
    test_ds,
    "indobert_imbalance",
)

all_results.append(res)
display(pd.DataFrame(all_results))



Eksperimen : indobert_imbalance
Train rows : 6,889
label
non-clickbait    4237
clickbait        2652


Map:   0%|          | 0/6889 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

{'loss': 0.7385, 'grad_norm': 6.377810001373291, 'learning_rate': 2.320185614849188e-06, 'epoch': 0.11600928074245939}
{'loss': 0.4223, 'grad_norm': 3.8474576473236084, 'learning_rate': 4.640371229698376e-06, 'epoch': 0.23201856148491878}
{'loss': 0.2497, 'grad_norm': 13.316307067871094, 'learning_rate': 6.960556844547565e-06, 'epoch': 0.3480278422273782}
{'loss': 0.2168, 'grad_norm': 21.129257202148438, 'learning_rate': 9.280742459396753e-06, 'epoch': 0.46403712296983757}
{'loss': 0.1956, 'grad_norm': 1.918036937713623, 'learning_rate': 1.160092807424594e-05, 'epoch': 0.580046403712297}
{'loss': 0.2161, 'grad_norm': 9.765225410461426, 'learning_rate': 1.392111368909513e-05, 'epoch': 0.6960556844547564}
{'loss': 0.1904, 'grad_norm': 0.5957614183425903, 'learning_rate': 1.624129930394432e-05, 'epoch': 0.8120649651972158}
{'loss': 0.2142, 'grad_norm': 9.398798942565918, 'learning_rate': 1.8561484918793505e-05, 'epoch': 0.9280742459396751}
{'eval_loss': 0.15848246216773987, 'eval_f1': 0.9

,experiment,model,train_rows,val_f1,test_precision,test_recall,test_f1,test_f1_macro
0,indobert_imbalance,indobenchmark/indobert-base-p1,6889,0.9523,0.9361,0.9362,0.936,0.9323


## 12. Eksperimen 2 — Balance

In [ ]:
res = train_and_evaluate(
    train_balance,
    val_ds,
    test_ds,
    "indobert_balance",
)

all_results.append(res)
display(pd.DataFrame(all_results))



Eksperimen : indobert_balance
Train rows : 8,474
label
clickbait        4237
non-clickbait    4237


Map:   0%|          | 0/8474 [00:00<?, ? examples/s]

{'loss': 0.7077, 'grad_norm': 4.789144515991211, 'learning_rate': 1.8867924528301889e-06, 'epoch': 0.09433962264150944}
{'loss': 0.4554, 'grad_norm': 5.081615447998047, 'learning_rate': 3.7735849056603777e-06, 'epoch': 0.18867924528301888}
{'loss': 0.2856, 'grad_norm': 10.112757682800293, 'learning_rate': 5.660377358490566e-06, 'epoch': 0.2830188679245283}
{'loss': 0.2495, 'grad_norm': 6.473211765289307, 'learning_rate': 7.5471698113207555e-06, 'epoch': 0.37735849056603776}
{'loss': 0.1854, 'grad_norm': 10.530723571777344, 'learning_rate': 9.433962264150944e-06, 'epoch': 0.4716981132075472}
{'loss': 0.2198, 'grad_norm': 14.045442581176758, 'learning_rate': 1.1320754716981132e-05, 'epoch': 0.5660377358490566}
{'loss': 0.2078, 'grad_norm': 8.489840507507324, 'learning_rate': 1.320754716981132e-05, 'epoch': 0.660377358490566}
{'loss': 0.1885, 'grad_norm': 3.3564436435699463, 'learning_rate': 1.5094339622641511e-05, 'epoch': 0.7547169811320755}
{'loss': 0.1743, 'grad_norm': 25.929620742797

,experiment,model,train_rows,val_f1,test_precision,test_recall,test_f1,test_f1_macro
0,indobert_imbalance,indobenchmark/indobert-base-p1,6889,0.9523,0.9361,0.9362,0.9360,0.9323
1,indobert_balance,indobenchmark/indobert-base-p1,8474,0.9593,0.9420,0.9420,0.9417,0.9383


## 13. Eksperimen 3 — Imbalance + Augment

In [ ]:
print("EDA pada imbalanced set...")

aug_texts_imb = eda_augment(train_imbalance["title_clean"].tolist())

df_aug_imb = train_imbalance.copy()
df_aug_imb["title_clean"] = aug_texts_imb

train_imb_aug = (
    pd.concat([train_imbalance, df_aug_imb])
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

print("[Imbalance + Augment] Distribusi:")
print(train_imb_aug[LABEL_COL].value_counts())

res = train_and_evaluate(
    train_imb_aug,
    val_ds,
    test_ds,
    "indobert_imbalance_augment",
)

all_results.append(res)
display(pd.DataFrame(all_results))


EDA pada imbalanced set...
[Imbalance + Augment] Distribusi:
label
non-clickbait    8474
clickbait        5304
Name: count, dtype: int64

Eksperimen : indobert_imbalance_augment
Train rows : 13,778
label
non-clickbait    8474
clickbait        5304


Map:   0%|          | 0/13778 [00:00<?, ? examples/s]

{'loss': 0.7792, 'grad_norm': 3.97767972946167, 'learning_rate': 1.1614401858304298e-06, 'epoch': 0.05803830528148578}
{'loss': 0.5242, 'grad_norm': 4.558851718902588, 'learning_rate': 2.3228803716608596e-06, 'epoch': 0.11607661056297155}
{'loss': 0.3503, 'grad_norm': 6.0098981857299805, 'learning_rate': 3.48432055749129e-06, 'epoch': 0.17411491584445735}
{'loss': 0.2475, 'grad_norm': 10.067021369934082, 'learning_rate': 4.645760743321719e-06, 'epoch': 0.2321532211259431}
{'loss': 0.2395, 'grad_norm': 2.0829427242279053, 'learning_rate': 5.8072009291521495e-06, 'epoch': 0.2901915264074289}
{'loss': 0.2283, 'grad_norm': 8.379976272583008, 'learning_rate': 6.96864111498258e-06, 'epoch': 0.3482298316889147}
{'loss': 0.2324, 'grad_norm': 16.9683837890625, 'learning_rate': 8.130081300813009e-06, 'epoch': 0.4062681369704005}
{'loss': 0.2039, 'grad_norm': 7.320624828338623, 'learning_rate': 9.291521486643439e-06, 'epoch': 0.4643064422518862}
{'loss': 0.2387, 'grad_norm': 4.285495758056641, 'l

,experiment,model,train_rows,val_f1,test_precision,test_recall,test_f1,test_f1_macro
0,indobert_imbalance,indobenchmark/indobert-base-p1,6889,0.9523,0.9361,0.9362,0.9360,0.9323
1,indobert_balance,indobenchmark/indobert-base-p1,8474,0.9593,0.9420,0.9420,0.9417,0.9383
2,indobert_imbalance_augment,indobenchmark/indobert-base-p1,13778,0.9583,0.9374,0.9374,0.9374,0.9339


## 14. Eksperimen 4 — Balance + Augment

In [ ]:
res = train_and_evaluate(
    train_augmented,
    val_ds,
    test_ds,
    "indobert_balance_augment",
)

all_results.append(res)
display(pd.DataFrame(all_results))



Eksperimen : indobert_balance_augment
Train rows : 16,948
label
clickbait        8474
non-clickbait    8474


Map:   0%|          | 0/16948 [00:00<?, ? examples/s]

{'loss': 0.7322, 'grad_norm': 5.398247718811035, 'learning_rate': 9.442870632672334e-07, 'epoch': 0.04719207173194903}
{'loss': 0.5675, 'grad_norm': 6.222331523895264, 'learning_rate': 1.8885741265344667e-06, 'epoch': 0.09438414346389806}
{'loss': 0.3737, 'grad_norm': 2.8345508575439453, 'learning_rate': 2.8328611898017e-06, 'epoch': 0.1415762151958471}
{'loss': 0.2847, 'grad_norm': 10.103473663330078, 'learning_rate': 3.7771482530689335e-06, 'epoch': 0.18876828692779613}
{'loss': 0.2665, 'grad_norm': 11.354190826416016, 'learning_rate': 4.721435316336166e-06, 'epoch': 0.23596035865974516}
{'loss': 0.2548, 'grad_norm': 10.006139755249023, 'learning_rate': 5.6657223796034e-06, 'epoch': 0.2831524303916942}
{'loss': 0.2403, 'grad_norm': 11.560373306274414, 'learning_rate': 6.6100094428706325e-06, 'epoch': 0.33034450212364325}
{'loss': 0.1821, 'grad_norm': 3.8122124671936035, 'learning_rate': 7.554296506137867e-06, 'epoch': 0.37753657385559225}
{'loss': 0.2165, 'grad_norm': 2.1894581317901

,experiment,model,train_rows,val_f1,test_precision,test_recall,test_f1,test_f1_macro
0,indobert_imbalance,indobenchmark/indobert-base-p1,6889,0.9523,0.9361,0.9362,0.9360,0.9323
1,indobert_balance,indobenchmark/indobert-base-p1,8474,0.9593,0.9420,0.9420,0.9417,0.9383
2,indobert_imbalance_augment,indobenchmark/indobert-base-p1,13778,0.9583,0.9374,0.9374,0.9374,0.9339
3,indobert_balance_augment,indobenchmark/indobert-base-p1,16948,0.9501,0.9466,0.9466,0.9464,0.9433


## 15. Ringkasan Hasil

In [ ]:
results_df = pd.DataFrame(all_results)

print("=" * 70)
print("HASIL AKHIR — IndoBERT Clickbait Detection")
print("=" * 70)

display(results_df)

results_path = RESULTS_DIR / "indobert_all_experiments.csv"
results_df.to_csv(results_path, index=False)

print(f"\nDisimpan ke: {results_path}")


HASIL AKHIR — IndoBERT Clickbait Detection


,experiment,model,train_rows,val_f1,test_precision,test_recall,test_f1,test_f1_macro
0,indobert_imbalance,indobenchmark/indobert-base-p1,6889,0.9523,0.9361,0.9362,0.9360,0.9323
1,indobert_balance,indobenchmark/indobert-base-p1,8474,0.9593,0.9420,0.9420,0.9417,0.9383
2,indobert_imbalance_augment,indobenchmark/indobert-base-p1,13778,0.9583,0.9374,0.9374,0.9374,0.9339
3,indobert_balance_augment,indobenchmark/indobert-base-p1,16948,0.9501,0.9466,0.9466,0.9464,0.9433



Disimpan ke: /content/drive/MyDrive/NLPAOL_V3/outputs_indobert/final_results/indobert_all_experiments.csv
